# CIS 5450 Project: Difficulty Topics
**Team Members:** [*Jiajun Chen, Pai Liang, Ruiyao Liu, Xinyi Gong*]

> This notebook documents how you implemented difficulty topics in your project. Use the link button in the top right when you select a cell to get a **hyperlink**.


---

## Topic: 1: **Near-zero-variance**
**Hyperlink: 4.1 Near-zero-variance**

### Implementation & Rationale


#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Near-zero-variance filtering is a preprocessing technique that removes features whose values are almost constant across all rows. In practice we found some features are effectively constant — for example, a binary flag that is `True` for 99% of rows. Such features carry virtually no discriminative information yet still inflate dimensionality, slow down downstream models, and can destabilize coefficient estimates in linear models. The standard approach is to compute the **dominant value ratio** (proportion of the most frequent value) for each column and drop any feature whose dominant value ratio exceeds a chosen threshold.

**How we implemented it:**
After excluding identifier columns (`appid`, `name`, `developer_name`, `publisher_name`) and the leakage column `estimated_sales`, we computed `X[col].value_counts(normalize=True).iloc[0]` for every feature and compared it against `DOMINANCE_THRESHOLD = 0.98`. Any column whose most-frequent value occupied ≥ 98% of all observations was dropped, with the column name and dominant ratio printed for transparency.

**Why we used it in this project:**
The Steam dataset contains many platform/flag columns (e.g., `supports_windows`, `supports_mac`, `supports_linux`) and category indicators that are extremely lopsided — `supports_windows` is `True` for nearly every game, so it cannot help discriminate high-revenue from low-revenue titles. Because our project's stated goal is to deliver **interpretable, actionable strategies for small game studios**, every retained feature must contribute real signal that a developer can act on. Removing NZV features early keeps the correlation heatmap (Step 4.2) and the LASSO regression (Step 4.3) focused on features that actually vary across the catalog, which directly improves the readability of the final coefficient ranking and avoids wasting model capacity on near-constant flags.

---

## Topic: 2: **Correlation Filtering**
**Hyperlink: 4.2 Correlation-based Filtering**

### Implementation & Rationale


#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Correlation filtering is a multicollinearity-reduction technique. When two features are highly linearly correlated, they encode largely the same information, which (i) inflates the variance of OLS / GLM coefficient estimates, (ii) makes the sign and magnitude of those coefficients unstable and hard to interpret, and (iii) can degrade the convergence and selection behavior of regularized methods such as LASSO. The standard procedure is to compute the Pearson correlation matrix, scan its upper triangle for pairs whose `|r|` exceeds a threshold (commonly `0.85`–`0.95`), and drop one feature from each redundant pair.

**How we implemented it:**
We computed `X.corr()`, masked the upper triangle with `np.triu(..., k=1)`, and collected every pair `(f1, f2)` with `|r| > CORR_THRESHOLD = 0.85`. Rather than dropping arbitrarily, for each pair we computed `corr(f1, y)` and `corr(f2, y)` against the log-revenue target and kept the feature with the stronger correlation to `y`, dropping the weaker one.

**Why we used it in this project.**
Steam metadata may have structurally redundant fields. If we left these in, the linear regression and statistical models in Part 5 would produce unstable, hard-to-explain coefficients. By keeping the more target-correlated feature from each pair, we preserve as much predictive signal as possible while making the downstream LASSO selection cleaner — LASSO is known to behave erratically on highly collinear inputs (it picks one feature from a correlated group somewhat arbitrarily), so pre-filtering at `|r| > 0.85` makes the final selected feature set much more reproducible and interpretable.

---

## Topic: 3: **Lasso**
**Hyperlink: 4.3 LASSO Filtering**

### Implementation & Rationale


#### How did you implement this topic and why did you use it?

**Brief Introduction:**
LASSO (Least Absolute Shrinkage and Selection Operator) is a linear regression method that augments the standard squared-error loss with an L1 penalty on the coefficient vector:

$$\min_{\beta} \frac{1}{2n} \|y - X\beta\|_2^2 + \alpha \|\beta\|_1$$

The defining property of the L1 penalty is that it can drive coefficients to exactly zero, so LASSO performs continuous shrinkage and discrete feature selection in a single optimization. The regularization strength `α` controls how aggressive that selection is: larger `α` yields a sparser, more parsimonious model.

**How we implemented it:**
We first standardized all surviving features with `StandardScaler` (zero mean, unit variance) so that the magnitudes of LASSO coefficients are directly comparable across features. We then fit `LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)`, which performs 5-fold cross-validation across 100 candidate `α` values and selects the one minimizing CV error. After fitting, we applied an additional pruning rule: any feature with `|coefficient| < 0.01` was treated as a negligible contributor and dropped. Finally, we plotted the surviving coefficients as a horizontal bar chart with **blue = positive effect on log-revenue** and **red = negative effect**, producing a one-glance ranking of which features actually push revenue up or down.

**Why we used it in this project.**
After the NZV and correlation steps, we still had a sizeable feature set, and we needed a principled, data-driven way to decide which features genuinely matter for predicting revenue. LASSO is ideal here because (a) it automatically performs feature selection through coefficient sparsity, (b) the cross-validated `α` removes the need to hand-tune the regularization strength, and (c) when fit on standardized inputs, the resulting coefficients double as a **direct importance ranking** — the bar chart we produced in Part 4.3 is essentially the actionable answer to *"which of these levers most strongly moves revenue?"* The resulting `X_selected` then serves as the cleaned, interpretable feature matrix that feeds directly into the OLS / statistical models in Part 5.